# mneme + AI Embeddings + Query Dashboard

This notebook shows an end-to-end real example:

1. Use a real embedding model (`sentence-transformers`) to generate vectors.
2. Store vectors in `mneme`.
3. Run semantic queries.
4. Visualize corpus + query results in a small dashboard.

> Tip: run this notebook from the repository root (`mneme-python`) so relative paths work.


In [ ]:
# If needed, install dependencies in your active kernel.
# You can skip this cell if these are already installed.

%pip install -q -e ../python sentence-transformers pandas scikit-learn plotly ipywidgets


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display

from mneme import Collection


In [ ]:
# 1) Meaningful corpus: AI/ML engineering knowledge base
#    (mimics docs/notes your team might actually search)
corpus = [
    "Embedding models convert text into dense vectors that preserve semantic similarity.",
    "Cosine similarity compares vector direction and is common for semantic retrieval.",
    "HNSW builds a graph index for approximate nearest-neighbor search in high dimensions.",
    "Retrieval-augmented generation combines vector search with LLM prompting.",
    "Use metadata filters to scope retrieval by tenant, product area, or timestamp.",
    "CI failures often come from import-time side effects and environment drift.",
    "Pin workflow actions and Python versions to reduce reproducibility issues in GitHub Actions.",
    "Use trusted publishing (OIDC) to release Python packages securely to PyPI.",
    "Mypy strict mode catches interface drift and unsafe assumptions early.",
    "Bandit helps detect insecure code patterns such as unsafe archive extraction.",
    "For production search, measure hit-rate@k and MRR in addition to latency.",
    "Normalize embeddings when using cosine similarity for stable ranking behavior.",
    "Use canary queries to monitor retrieval quality regressions after model updates.",
    "Avoid leaking file handles or native resources by using context managers.",
    "Cache model artifacts and native libraries in CI to avoid rate-limit flakiness.",
]

df = pd.DataFrame(
    {
        "id": [f"doc_{i}" for i in range(len(corpus))],
        "text": corpus,
        "topic": [
            "embeddings",
            "embeddings",
            "ann-index",
            "rag",
            "retrieval",
            "ci",
            "ci",
            "release",
            "typing",
            "security",
            "evaluation",
            "embeddings",
            "monitoring",
            "runtime",
            "ci",
        ],
    }
)

df.head()


In [ ]:
# 2) Build embeddings with a real AI model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

embeddings = model.encode(df["text"].tolist(), normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype=np.float32)

dimension = int(embeddings.shape[1])
print(f"Model: {model_name}")
print(f"Embeddings shape: {embeddings.shape}")


In [ ]:
# 3) Create mneme collection and insert vectors
#    (in-memory collection for this demo)
collection = Collection("notebook_demo", dimension=dimension)

for i, row in df.iterrows():
    collection.insert(row["id"], embeddings[i], metadata=row["text"])

print("Inserted rows:", collection.count())


In [ ]:
# Optional: persist and reload to show save/load path
save_path = Path("./notebook_demo.mneme")
collection.save(save_path)

loaded = Collection.load(save_path)
print("Reloaded rows:", loaded.count())


In [ ]:
# 4) Query helper
id_to_text = dict(zip(df["id"], df["text"]))

def semantic_search(query: str, k: int = 5):
    q = model.encode([query], normalize_embeddings=True)
    q = np.asarray(q, dtype=np.float32)[0]
    results = loaded.search(q, k)

    rows = []
    for r in results:
        rows.append({"id": r.id, "score": float(r.score), "text": id_to_text.get(r.id, "")})
    return pd.DataFrame(rows)

semantic_search("How do I debug GitHub Actions CI failures?", k=5)


In [ ]:
# 5) Quantitative retrieval quality check (simple offline eval)
#    We define expected "gold" docs for a few realistic queries.

eval_queries = [
    {
        "query": "How can we make CI less flaky when downloading artifacts?",
        "gold_ids": {"doc_14", "doc_6", "doc_5"},
    },
    {
        "query": "What metric should we use for semantic ranking quality?",
        "gold_ids": {"doc_10", "doc_0", "doc_1"},
    },
    {
        "query": "How does HNSW relate to nearest-neighbor retrieval?",
        "gold_ids": {"doc_2", "doc_3"},
    },
]


def evaluate_retrieval(k: int = 5) -> pd.DataFrame:
    rows = []
    for case in eval_queries:
        result_df = semantic_search(case["query"], k=k)
        retrieved = result_df["id"].tolist()
        gold = case["gold_ids"]

        # hit@k: at least one relevant result
        hit = any(doc_id in gold for doc_id in retrieved)

        # reciprocal rank of first relevant result
        rr = 0.0
        for rank, doc_id in enumerate(retrieved, start=1):
            if doc_id in gold:
                rr = 1.0 / rank
                break

        rows.append(
            {
                "query": case["query"],
                "k": k,
                "hit@k": float(hit),
                "rr": rr,
                "top_ids": retrieved,
            }
        )

    out_df = pd.DataFrame(rows)
    print(f"Average hit@{k}: {out_df['hit@k'].mean():.3f}")
    print(f"MRR@{k}: {out_df['rr'].mean():.3f}")
    return out_df


evaluate_retrieval(k=5)


In [ ]:
# 5) Build a 2D projection dashboard (corpus + query vector)
#    This is a lightweight dashboard using Plotly + ipywidgets.

pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(embeddings)

plot_df = df.copy()
plot_df["x"] = xy[:, 0]
plot_df["y"] = xy[:, 1]

query_box = widgets.Text(
    value="vector search and embeddings",
    description="Query:",
    layout=widgets.Layout(width="75%"),
)

k_slider = widgets.IntSlider(value=5, min=1, max=min(10, len(df)), step=1, description="Top-k")
out = widgets.Output()


def render_dashboard(*_args):
    with out:
        out.clear_output(wait=True)

        query = query_box.value.strip()
        q = model.encode([query], normalize_embeddings=True)
        q = np.asarray(q, dtype=np.float32)[0]

        q_xy = pca.transform(q.reshape(1, -1))[0]
        res_df = semantic_search(query, k=int(k_slider.value))
        top_ids = set(res_df["id"].tolist())

        draw_df = plot_df.copy()
        draw_df["is_top_hit"] = draw_df["id"].isin(top_ids)

        fig = px.scatter(
            draw_df,
            x="x",
            y="y",
            color="is_top_hit",
            hover_data=["id", "text"],
            title=f"Semantic space (top-{k_slider.value} hits highlighted)",
            color_discrete_map={True: "crimson", False: "steelblue"},
        )

        fig.add_scatter(
            x=[q_xy[0]],
            y=[q_xy[1]],
            mode="markers+text",
            name="query",
            text=["query"],
            textposition="top center",
            marker=dict(size=16, color="black", symbol="x"),
        )

        fig.update_layout(height=520)
        fig.show()

        display(res_df)


query_box.observe(render_dashboard, names="value")
k_slider.observe(render_dashboard, names="value")

display(widgets.VBox([query_box, k_slider, out]))
render_dashboard()


In [ ]:
# 6) Cleanup (optional)
# Run this when done to release handles.

# loaded.close()
# collection.close()
